# get_features_cat_regression

Esta función recibe como argumentos un dataframe, el nombre de una de las columnas del mismo (argumento 'target_col'), que debería ser el target de un hipotético modelo de regresión, es decir debe ser una variable numérica continua o discreta pero con alta cardinalidad y una variable float "pvalue" cuyo valor por defecto será 0.05.

La función debe devolver una lista con las columnas categóricas del dataframe cuyo test de relación con la columna designada por 'target_col' supere en confianza estadística el test de relación que sea necesario hacer (es decir la función debe poder escoger cuál de los dos test que hemos aprendido tiene que hacer).

La función debe hacer todas las comprobaciones necesarias para no dar error como consecuecia de los valores de entrada. Es decir hará un check de los valores asignados a los argumentos de entrada y si estos no son adecuados debe retornar None y printar por pantalla la razón de este comportamiento. Ojo entre las comprobaciones debe estar que "target_col" hace referencia a una variable numérica continua del dataframe.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats


def get_features_cat_regression(df, target_col, pvalue = 0.05):
    """
Filtra columnas categóricas que tienen relación estadísticamente significativa
con la columna target (numérica).

Parámetros:
-----------
df : DataFrame
    El dataframe con los datos
target_col : str
    Nombre de la columna target (numérica continua o discreta con alta cardinalidad)
alpha : float
    Nivel de significancia (por defecto 0.05)

Retorna:
--------
list : Lista de nombres de columnas categóricas significativas
"""
    # Comprobación de los argumentos de entrada:
    if not isinstance(df, pd.DataFrame):
        print("Error: El argumento 'df' debe ser un DataFrame")
        return None
    
    if df[target_col].dtypes in ["object","category"]:
        print("El target no es válido, tiene que ser una variable numérica")
        return None
    cardinalidad = len(df[target_col].unique())/len(df)
    if cardinalidad < 0.30:
        print(f"La variable target tiene baja cardinalidad,{cardinalidad :2%}, necesita alta cardinalidad para regresión")
        return None
     # target_col == "numerica_continua" | (target_col == "numerica_discreta" & target_col == "alta_cardinalidad")
    
    
    # Creamos la lista para las categóricas válidas, la lista que nos va a devolver la función
    lista_col_categoricas_significativas= []
    
    # Creamos la lista de categóricas para pasar el test.
    col_categoricas = []
    for col in df.select_dtypes(include=["object", "category"]).columns:
        if df[col].nunique() < 10:  # Menos de 10 valores únicos
            col_categoricas.append(col)
    # Seleccionamos el test
    for col in col_categoricas:
        grupos= df[col].unique()
        n_grupos =len(df[col].unique())
        if n_grupos == 2: # Para categórica binaria
            cat1, cat2 = df[col].unique()  # Desempaquetar las dos categorías
            a = df[df[col] == cat1][target_col]
            b = df[df[col] == cat2][target_col]
            _ , p_val = stats.ttest_ind( a , b) # t test
        
        elif n_grupos > 2: # Para categórica no binaria
            taldea = [df[df[col]==grupo][target_col]for grupo in grupos]

            _ , p_val = stats.f_oneway(*taldea) # Test ANOVA
        else:  # Si no sirve la variable para ningún test
            continue    

        if p_val < pvalue:
                    lista_col_categoricas_significativas.append(col)



    return lista_col_categoricas_significativas

In [ ]:
# Seleccionamos el test
for col in col_categoricas:
    grupos= df[col].unique()
    n_grupos =len(df[col].unique())
    if n_grupos == 2: # Para categórica binaria
        cat1, cat2 = df[col].unique()  # Desempaquetar las dos categorías
        a = df[df[col] == cat1][target_col]
        b = df[df[col] == cat2][target_col]
        _ , p_val = stats.ttest_ind( a , b) # t test
    
    elif n_grupos > 2: # Para categórica no binaria
        taldea = [df[df[col]==grupo][target_col]for grupo in grupos]

        _ , p_val = stats.f_oneway(*taldea) # Test ANOVA
    else:  # Si no sirve la variable para ningún test
        continue    

    if p_val < pvalue:
                lista_col_categoricas_significativas.append(col)

